# AgenticEd Labs · Week 2 — LLM Engineering: Tokens, Sampling, Structure, and Evaluation

<a href="https://colab.research.google.com/github/AgenticEdOrg/AgenticEd/blob/main/labs/week2_llm_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

| | |
|---|---|
| **Time** | 75–90 minutes |
| **Approx. API cost** | ≈ US$0.10 with the default low-cost model |
| **Prerequisites** | Week 1 |
| **You'll use** | tiktoken, Hugging Face transformers (GPT-2, runs locally), Pydantic AI, pandas |

**Learning objectives**
- Count tokens and estimate cost and context usage before you send a request
- Implement **temperature, top-k and top-p sampling** yourself on a real model's logits
- Get **validated structured output** with automatic repair on failure
- **Evaluate prompts** on a labeled dataset (accuracy, confusion matrix, consistency) instead of eyeballing
- Measure **hallucination and abstention** rates

> **Before you start:** add `OPENAI_API_KEY` *or* `ANTHROPIC_API_KEY` to Colab Secrets (key icon in the left sidebar → enable notebook access), or to `labs/.env` locally. See `00_environment_check.ipynb` if anything fails.

In [ ]:
# ── Setup (run once) ─────────────────────────────────────────────────────────
# In Colab: clones the course repo and installs the pinned requirements (2–3 minutes the first time).
# If Colab asks you to restart the runtime after installing, do so and run this cell again.
import sys, pathlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not pathlib.Path("/content/AgenticEd").exists():
        !git clone -q --depth 1 https://github.com/AgenticEdOrg/AgenticEd.git /content/AgenticEd
    %cd /content/AgenticEd/labs
    %pip install -q -r requirements.txt
sys.path.insert(0, str(pathlib.Path.cwd()))

import labkit
cfg = labkit.setup()          # reads your key, picks the model, prints what it chose

## 1 · Tokens are the unit of cost, latency, and limits

`tiktoken` implements the tokenizers OpenAI models use (`o200k_base` for current models). Other providers' tokenizers differ, but the lessons transfer: rare words split into pieces, and non-English text often costs several times more.

In [ ]:
import tiktoken, pandas as pd
enc = tiktoken.get_encoding("o200k_base")
samples = {
    "English": "Agents plan, call tools, observe results, and adapt until the goal is met.",
    "Spanish": "Los agentes planifican, usan herramientas, observan resultados y se adaptan hasta cumplir el objetivo.",
    "Hindi": "एजेंट योजना बनाते हैं, उपकरणों का उपयोग करते हैं, परिणाम देखते हैं और लक्ष्य पूरा होने तक अनुकूलन करते हैं।",
    "Tamil": "முகவர்கள் திட்டமிட்டு, கருவிகளைப் பயன்படுத்தி, முடிவுகளைக் கவனித்து, இலக்கை அடையும் வரை மாற்றிக்கொள்கின்றன.",
    "JSON": '{"tool": "get_forecast", "arguments": {"city": "Chennai", "days": 3}}',
}
rows = [{"text": k, "characters": len(v), "words": len(v.split()), "tokens": len(enc.encode(v))} for k, v in samples.items()]
df = pd.DataFrame(rows); df["tokens_per_word"] = (df.tokens / df.words).round(2)
display(df)
print("English pieces:", [enc.decode([t]) for t in enc.encode(samples["English"])][:16])

In [ ]:
def estimate(messages, expected_output_tokens=300, context_limit=128_000):
    """Pre-flight check: token count, share of the context window, and rough cost."""
    n_in = sum(len(enc.encode(m["content"])) + 4 for m in messages)      # ~4 tokens of per-message overhead
    return {"input_tokens": n_in, "context_used_pct": round(100 * (n_in + expected_output_tokens) / context_limit, 2),
            "est_cost_usd": cfg.cost(n_in, expected_output_tokens)}

corpus_text = "\n\n".join(d["text"] for d in labkit.load_corpus())
print("Whole policy corpus as one prompt:", estimate([{"role": "user", "content": corpus_text}]))

## 2 · Inside the model: logits and sampling

API models hide their internals (and current reasoning models don't expose temperature at all), so we open a small open model — **GPT-2**, 124M parameters, runs on CPU — and implement sampling ourselves. A model outputs one score (*logit*) per vocabulary token; everything else is your decoding strategy.

In [ ]:
import torch, numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2 = AutoModelForCausalLM.from_pretrained("gpt2").eval()

def next_logits(prompt):
    ids = gpt2_tok(prompt, return_tensors="pt").input_ids
    with torch.no_grad():
        return gpt2(ids).logits[0, -1].numpy()

def softmax(x):
    e = np.exp(x - x.max()); return e / e.sum()

logits = next_logits("The safest way to deploy an AI agent is to")
top = np.argsort(-logits)[:8]
display(pd.DataFrame({"token": [gpt2_tok.decode([i]) for i in top], "logit": logits[top].round(2),
                      "p(T=1)": softmax(logits)[top].round(3), "p(T=0.5)": softmax(logits / 0.5)[top].round(3),
                      "p(T=1.5)": softmax(logits / 1.5)[top].round(3)}))

In [ ]:
rng = np.random.default_rng(0)

def sample(logits, temperature=1.0, top_k=None, top_p=None):
    """Return one token id. temperature<=0 means greedy."""
    if temperature <= 0:
        return int(np.argmax(logits))
    probs = softmax(logits / temperature)
    if top_k:                                    # keep only the k most likely tokens
        cutoff = np.sort(probs)[-top_k]
        probs = np.where(probs >= cutoff, probs, 0)
    if top_p:                                    # nucleus sampling (exercise below asks you to write this)
        probs = apply_top_p(probs, top_p)
    probs = probs / probs.sum()
    return int(rng.choice(len(probs), p=probs))

def apply_top_p(probs, p):
    # TODO (exercise): keep the smallest set of tokens whose cumulative probability >= p; zero the rest
    raise NotImplementedError

def generate(prompt, n=20, **kw):
    ids = gpt2_tok(prompt).input_ids
    for _ in range(n):
        with torch.no_grad():
            lg = gpt2(torch.tensor([ids])).logits[0, -1].numpy()
        ids.append(sample(lg, **kw))
    return gpt2_tok.decode(ids)

for settings in [dict(temperature=0), dict(temperature=0.7, top_k=40), dict(temperature=1.6)]:
    print(settings, "→", generate("The safest way to deploy an AI agent is to", **settings).replace("\n", " "))

In [ ]:
import matplotlib.pyplot as plt
temps = np.linspace(0.2, 2.0, 19)
entropy = [-(softmax(logits / t) * np.log(softmax(logits / t) + 1e-12)).sum() for t in temps]
plt.figure(figsize=(7, 3)); plt.plot(temps, entropy, marker="o", color="#6D28D9")
plt.xlabel("temperature"); plt.ylabel("entropy (nats)"); plt.title("Higher temperature → flatter distribution → less predictable output")
plt.tight_layout(); plt.show()

### Exercise — implement nucleus (top-p) sampling

Implement `apply_top_p` above, re-run that cell, then run the check.

In [ ]:
try:
    p = np.array([0.5, 0.3, 0.15, 0.05])
    out = apply_top_p(p.copy(), 0.8)
    assert np.allclose(out, [0.5, 0.3, 0, 0]), f"expected [0.5, 0.3, 0, 0], got {out}"
    print("✓ Exercise passed —", generate("The best way to learn is", temperature=0.9, top_p=0.9))
except NotImplementedError:
    print("Not done yet — implement apply_top_p.")
except AssertionError as e:
    print("Not quite:", e)

<details><summary><b>Show a solution</b></summary>

```python
def apply_top_p(probs, p):
    order = np.argsort(-probs)
    cumulative = np.cumsum(probs[order])
    keep = order[: int(np.searchsorted(cumulative, p) + 1)]
    out = np.zeros_like(probs); out[keep] = probs[keep]
    return out
```

</details>

## 3 · Structured output you can trust

Downstream code needs typed data, not prose. Pydantic AI sends your model's JSON Schema, **validates** the reply, and on failure sends the validation error back for the model to fix. An `output_validator` adds business rules that a schema can't express.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field
from pydantic_ai import Agent, ModelRetry

class TicketTriage(BaseModel):
    category: Literal["billing", "technical", "account", "policy", "other"]
    urgency: Literal["low", "medium", "high"]
    summary: str = Field(max_length=120)
    needs_human: bool = Field(description="True for anything involving money, safety, or personal data")

triage_agent = Agent(cfg.pydantic_ai, output_type=TicketTriage,
                     instructions="Triage support tickets for a school district's IT and admin help desk.")

@triage_agent.output_validator
def safety_rule(output: TicketTriage) -> TicketTriage:
    if output.category == "billing" and not output.needs_human:
        raise ModelRetry("Billing tickets always need a human. Set needs_human=true.")
    return output

r = await triage_agent.run("I was charged twice for my son's laptop repair and need the money back before Friday.")
print(r.output.model_dump_json(indent=2))

## 4 · Evaluate prompts, don't eyeball them

A prompt that works on one example proves nothing. Below: 24 labeled help-desk tickets, three prompt variants, and real metrics. This is the habit that separates prototypes from production.

In [ ]:
LABELED = [
    ("My laptop screen is cracked after it fell off my desk.", "technical"),
    ("I can't sign in, it says my account is locked.", "account"),
    ("Why was I charged $25 for a repair?", "billing"),
    ("Can students use AI tools during the mid-term exams?", "policy"),
    ("The Wi-Fi in Room 214 keeps dropping.", "technical"),
    ("I forgot my password and the reset email never arrives.", "account"),
    ("Is there a refund if the repair was the school's fault?", "billing"),
    ("Who can I ask about what data an AI tool stored about my child?", "policy"),
    ("The projector in the library won't turn on.", "technical"),
    ("Please change my email address on my student account.", "account"),
    ("I paid the repair fee twice by mistake.", "billing"),
    ("Is NotebookLM approved for students under 13?", "policy"),
    ("When does the robotics club meet?", "other"),
    ("Chromebook keyboard is missing two keys.", "technical"),
    ("My account was disabled after too many attempts.", "account"),
    ("Can the repair fee be waived for families with financial need?", "billing"),
    ("Do I have to say when I used AI in my essay?", "policy"),
    ("Where is the lost-and-found?", "other"),
    ("My laptop won't charge even with a new cable.", "technical"),
    ("I need two-factor authentication reset on my account.", "account"),
    ("How do I pay for a replacement charger?", "billing"),
    ("What happens if I'm caught using AI on homework?", "policy"),
    ("Is the activity bus running on Wednesdays?", "other"),
    ("The printer says it's offline for everyone in Room 108.", "technical"),
]
LABELS = ["billing", "technical", "account", "policy", "other"]

class Label(BaseModel):
    category: Literal["billing", "technical", "account", "policy", "other"]

PROMPTS = {
    "v1 bare": "Classify the help-desk ticket.",
    "v2 definitions": ("Classify the help-desk ticket. billing = fees, payments, refunds. technical = devices, network, hardware. "
                       "account = sign-in, passwords, account settings. policy = rules about AI use, data, exams, integrity. "
                       "other = anything else."),
    "v3 definitions + examples": ("Classify the help-desk ticket. billing = fees, payments, refunds. technical = devices, network, hardware. "
                                  "account = sign-in, passwords, account settings. policy = rules about AI use, data, exams, integrity. "
                                  "other = anything else.\nExamples: 'Refund my charger fee' -> billing; 'Laptop fan is loud' -> technical; "
                                  "'Reset my password' -> account; 'Can I use AI in class?' -> policy; 'Cafeteria hours?' -> other."),
}

import asyncio
async def evaluate_prompt(name, instructions):
    agent = Agent(cfg.pydantic_ai, output_type=Label, instructions=instructions)
    async def one(text):
        try:
            return (await agent.run(text)).output.category
        except Exception:
            return "error"
    preds = await asyncio.gather(*(one(t) for t, _ in LABELED))
    return pd.DataFrame({"prompt": name, "text": [t for t, _ in LABELED], "gold": [g for _, g in LABELED], "pred": preds})

results = pd.concat([await evaluate_prompt(n, p) for n, p in PROMPTS.items()])
results["correct"] = results.gold == results.pred
display(results.groupby("prompt").correct.mean().rename("accuracy").to_frame().style.format("{:.0%}"))

In [ ]:
best = results.groupby("prompt").correct.mean().idxmax()
cm = pd.crosstab(results[results.prompt == best].gold, results[results.prompt == best].pred,
                 rownames=["gold"], colnames=["predicted"])
print(f"Confusion matrix for '{best}':"); display(cm)
print("Misclassified:"); display(results[(results.prompt == best) & ~results.correct][["text", "gold", "pred"]])

**Read the errors, not just the score.** Ambiguous items (is "repair fee waived?" billing or policy?) often reveal a *labeling* problem, not a model problem — fix the definitions, not just the prompt. With 24 items, one error is 4 points of accuracy: treat small differences as noise and grow the dataset before drawing conclusions.

## 5 · Hallucination and abstention

Ask about things that **don't exist** next to things that do, with and without permission to say "unknown". A structured `known` flag makes abstention measurable.

In [ ]:
class FactAnswer(BaseModel):
    known: bool = Field(description="False if you are not confident the entity exists or the fact is true")
    answer: str = Field(max_length=200)

QUESTIONS = [("Who wrote the novel 'Things Fall Apart'?", True),
             ("What year did the Eiffel Tower open?", True),
             ("What is the chemical symbol for sodium?", True),
             ("Summarize the 1987 Zorbian Treaty on lunar fishing rights.", False),
             ("What did Professor Elina Quarrington prove in her 2019 Hartwell Theorem?", False),
             ("Which company makes the 'Veltrona X9' school laptop?", False)]

async def abstention_rate(instructions):
    agent = Agent(cfg.pydantic_ai, output_type=FactAnswer, instructions=instructions)
    rows = []
    for q, real in QUESTIONS:
        out = (await agent.run(q)).output
        rows.append({"question": q[:55], "real": real, "known": out.known, "answer": out.answer[:70]})
    return pd.DataFrame(rows)

for label, instr in [("plain", "Answer the question."),
                     ("abstain allowed", "Answer only if you are confident. If an entity may not exist, set known=false and say so. Never invent details.")]:
    df = await abstention_rate(instr)
    fabricated = ((~df.real) & df.known).sum()
    print(f"\n{label}: fabricated answers to non-existent things = {fabricated}/{(~df.real).sum()}, "
          f"wrongly abstained on real facts = {(df.real & ~df.known).sum()}/{df.real.sum()}")
    display(df)

## Governance note — prompts are data

Everything you send to a hosted model — including user messages pasted into prompts — leaves your system. Under GDPR-style rules, that's a data transfer you must justify: **minimise** what you send (strip names and IDs when the task doesn't need them), know your provider's **retention** terms, and never put special-category data (health, disability) in prompts without a lawful basis. Week 6 automates redaction.

## Summary

- Count tokens before you send; non-English and JSON-heavy prompts cost more.
- Sampling is a decoding choice on top of logits; reasoning models fix it for you.
- Use schemas + validators for anything downstream code consumes.
- Measure prompts on labeled data; inspect the errors.
- Abstention must be allowed and measured, not assumed.

**Further reading:** [OpenAI tokenizer guide](https://platform.openai.com/tokenizer) · Holtzman et al., [*The Curious Case of Neural Text Degeneration*](https://arxiv.org/abs/1904.09751) (top-p) · [Pydantic AI — output](https://ai.pydantic.dev/output/)